In [1]:
!pip install huggingface_hub

In [2]:
import os
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_API_TOKEN")
os.environ["HF_TOKEN"] = user_secrets.get_secret("kaggle_hf_key")
api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")

Logged in as: equalNull


## find the files common every day
let us find the hosts present in every day.

In [3]:
import re
from collections import Counter
from huggingface_hub import HfApi

api = HfApi()
total_counts = Counter()

# 1. Iterate through all 10 repositories
for day in range(16, 26):
    repo_id = f"equalNull/labelled-optc-sep{day}"
    print(f"Scanning {repo_id}...")
    
    try:
        # Get the list of all file names in the repo
        files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
        
        # 2. Extract 4-digit numbers from filenames
        for filename in files:
            # Finds all 4-digit sequences in the filename
            matches = re.findall(r'\d{4}', filename)
            for match in matches:
                # Only count if it's within your 0000-1000 range
                if 0 <= int(match) <= 1000:
                    total_counts[match] += 1
                    
    except Exception as e:
        print(f"Skipping {repo_id} due to error: {e}")


Scanning equalNull/labelled-optc-sep16...
Scanning equalNull/labelled-optc-sep17...
Scanning equalNull/labelled-optc-sep18...
Scanning equalNull/labelled-optc-sep19...
Scanning equalNull/labelled-optc-sep20...
Scanning equalNull/labelled-optc-sep21...
Scanning equalNull/labelled-optc-sep22...
Scanning equalNull/labelled-optc-sep23...
Scanning equalNull/labelled-optc-sep24...
Scanning equalNull/labelled-optc-sep25...


In [4]:
# Filter for IDs that appeared in all 10 repositories
all_10_days = [p for p, count in total_counts.items() if count == 10]
print(all_10_days)
print(len(all_10_days))

['0101', '0102', '0103', '0104', '0105', '0106', '0107', '0108', '0109', '0110', '0111', '0112', '0113', '0114', '0115', '0116', '0117', '0118', '0119', '0120', '0121', '0122', '0123', '0124', '0125', '0151', '0152', '0153', '0154', '0155', '0156', '0157', '0158', '0159', '0160', '0161', '0162', '0163', '0164', '0165', '0166', '0167', '0168', '0169', '0170', '0171', '0172', '0173', '0174', '0175', '0201', '0202', '0203', '0204', '0205', '0206', '0207', '0208', '0209', '0210', '0211', '0212', '0214', '0215', '0216', '0217', '0218', '0219', '0220', '0221', '0223', '0224', '0225', '0251', '0252', '0253', '0254', '0255', '0256', '0257', '0258', '0259', '0260', '0261', '0262', '0263', '0264', '0265', '0266', '0267', '0268', '0269', '0270', '0271', '0272', '0273', '0274', '0275', '0301', '0302', '0303', '0304', '0305', '0306', '0307', '0308', '0309', '0310', '0311', '0312', '0313', '0314', '0315', '0316', '0317', '0318', '0319', '0320', '0321', '0322', '0323', '0324', '0325', '0351', '0352',

## hosts attacked during attack phase
```python
mal_hosts = {0609, 0874, 0955, 0201, 0660, 0104, 0321, 0355, 0205, 0255, 0402, 0419, 0462, 0771, 0170, 0503, 0559}
```
The proof of this can be found [here](https://www.kaggle.com/code/faihaj/labelling-optc-dataset?scriptVersionId=306235992).

## generate names of random host patterns

In [5]:
import random

# 1. Start with known malicious hosts
hosts = {"0609", "0874", "0955", "0201", "0660", "0104",
         "0321", "0355", "0205", "0255", "0402", "0419",
         "0462", "0771", "0170", "0503", "0559"}

# 2. Identify how many more you need to reach 100
needed = 100 - len(hosts)

# 3. Filter all_10_days to exclude what's already in hosts
candidates = [p for p in all_10_days if p not in hosts]

# 4. Randomly pick from the persistent set instead of a random range
if len(candidates) >= needed:
    new_hosts = random.sample(candidates, needed)
else:
    # Fallback if there aren't enough persistent IDs
    print(f"Warning: Only {len(candidates)} candidates found in all_10_days. Using all of them.")
    new_hosts = candidates

# 5. Combine and create patterns
hosts.update(new_hosts)
host_patterns = {f"*{h}*" for h in hosts}

print(f"Final host count: {len(hosts)}")


Final host count: 100


In [6]:
print(host_patterns)

{'*0422*', '*0609*', '*0223*', '*0405*', '*0460*', '*0268*', '*0207*', '*0253*', '*0264*', '*0375*', '*0955*', '*0259*', '*0255*', '*0170*', '*0062*', '*0061*', '*0418*', '*0461*', '*0466*', '*0402*', '*0158*', '*0201*', '*0156*', '*0313*', '*0252*', '*0408*', '*0110*', '*0463*', '*0266*', '*0254*', '*0307*', '*0771*', '*0107*', '*0403*', '*0214*', '*0215*', '*0874*', '*0469*', '*0473*', '*0318*', '*0453*', '*0301*', '*0319*', '*0425*', '*0353*', '*0174*', '*0467*', '*0257*', '*0321*', '*0368*', '*0312*', '*0351*', '*0175*', '*0104*', '*0462*', '*0208*', '*0401*', '*0113*', '*0470*', '*0211*', '*0258*', '*0063*', '*0059*', '*0357*', '*0111*', '*0355*', '*0202*', '*0068*', '*0267*', '*0320*', '*0225*', '*0310*', '*0559*', '*0114*', '*0354*', '*0218*', '*0366*', '*0261*', '*0071*', '*0503*', '*0411*', '*0162*', '*0065*', '*0260*', '*0269*', '*0070*', '*0205*', '*0212*', '*0364*', '*0660*', '*0323*', '*0122*', '*0369*', '*0220*', '*0314*', '*0420*', '*0362*', '*0217*', '*0161*', '*0419*'}

## Download the parquets
We will download all the hosts parquet for each day during `16th of september` to `25th of september`.
## upload into hf dataset

In [ ]:
from huggingface_hub import snapshot_download

new_repo_id = f"equalNull/hosts100-labelled-optc"
api.create_repo(repo_id=new_repo_id, repo_type="dataset", exist_ok=True)
working_path = "/kaggle/temp/"
os.makedirs(working_path, exist_ok=True)
for day in range(16,26):
    repo_id = f"equalNull/labelled-optc-sep{day}"
    downloaded_path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=working_path,
        allow_patterns=list(host_patterns),
        max_workers=4
    )
    
    !mkdir -p {working_path}data/sep{day}
    !mv {working_path}data/*.parquet {working_path}data/sep{day}/
    api.upload_large_folder(
        folder_path=f"{working_path}data/",
        repo_id=new_repo_id,
        repo_type="dataset"
    )
    !rm -rf {working_path}data/

Fetching 93 files:   0%|          | 0/93 [00:00<?, ?it/s]

Recovering from metadata files:   0%|          | 0/93 [00:00<?, ?it/s]




---------- 2026-03-26 15:21:37 (0:00:00) ----------
Files:   hashed 0/93 (0.0/1.8G) | pre-uploaded: 0/0 (0.0/1.8G) (+93 unsure) | committed: 0/93 (0.0/1.8G) | ignored: 0
Workers: hashing: 2 | get upload mode: 0 | pre-uploading: 0 | committing: 0 | waiting: 0
---------------------------------------------------


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            


---------- 2026-03-26 15:56:32 (0:02:00) ----------
Files:   hashed 100/100 (12.5G/12.5G) | pre-uploaded: 100/100 (12.5G/12.5G) | committed: 0/100 (0.0/12.5G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 0 | committing: 1 | waiting: 1
---------------------------------------------------
                             

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Fetching 100 files:   0%|          | 0/100 [00:00<?, ?it/s]